# Convert Zod Frame Dataset to Tabular

## Initialization

In [1]:
import sys
import pandas as pd
from typing import List
import numpy as np
from matplotlib import pyplot as plt
import random

from zod import ZodFrames
from zod import ZodSequences
import zod.constants as constants
from zod.constants import Camera, Lidar, Anonymization, AnnotationProject
from zod.data_classes import LidarData
from zod.visualization.oxts_on_image import visualize_oxts_on_image
from zod import ObjectAnnotation
from zod.visualization.object_visualization import overlay_object_2d_box_on_image, overlay_object_3d_box_on_image
from zod.visualization.lidar_on_image import visualize_lidar_on_image
from zod import EgoRoadAnnotation
from zod.utils.polygon_transformations import polygons_to_binary_mask
from zod.utils.polygon_transformations import polygons_to_binary_mask
from zod.visualization.polygon_utils import overlay_mask_on_image
from zod import LaneAnnotation

import openmeteo_requests
import openmeteo_requests
import requests_cache
from retry_requests import retry


sys.path.append("../")
plt.rcParams["figure.figsize"] = [20, 10]

dataset_root = "../data/zod"  
version = "mini"  
zod_frames = ZodFrames(dataset_root=dataset_root, version=version)

/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Split into Training and Validation sets

In [2]:
# get default training and validation splits
training_frames = zod_frames.get_split(constants.TRAIN)
validation_frames = zod_frames.get_split(constants.VAL)

# print the number of training and validation frames
print(f"Number of training frames: {len(training_frames)}")
print(f"Number of validation frames: {len(validation_frames)}")

# print out the first 5 training frames
print("The 5 first training frames have the ids:", sorted(list(training_frames))[:5])

Number of training frames: 10
Number of validation frames: 2
The 5 first training frames have the ids: ['009158', '018591', '023996', '029229', '044953']


## Features

### Metadata

Possible features:
- frame_id
- country_code
- time_of_day
- Image annotations: collection_car, num_traffic_light, num_traffic_sign, num_pedestrians, num_lane_pedestrian, num_vehicles, num_vulnerable_vehicles, num_lane_instances
- longitud, lattitude
- Enviromental proxies: road_condition, road_type, scrapped_weather, solar_angle_elevation

**ID**

Not usefull per se but will be used for identification.

In [3]:
frames_id = [zod_frames[frame_id].metadata.frame_id for frame_id in training_frames]
print(frames_id)

['009158', '029229', '082291', '087912', '065838', '083430', '044953', '018591', '023996', '070221']


**Country**

Not super usefull but could be analyzed. 

In [4]:
country = [zod_frames[frame_id].metadata.country_code for frame_id in training_frames]
print(country)

['DE', 'PL', 'PL', 'DE', 'DE', 'PL', 'SE', 'IT', 'PL', 'DE']


**Time of Day**

Differenciates between day/night. Maybe a more fine grained analysis, with morining, afteroon, evening and night could be more useful. 

In [5]:
time_of_day = [zod_frames[frame_id].metadata.time_of_day for frame_id in training_frames]
print(time_of_day)

['day', 'day', 'night', 'day', 'day', 'night', 'day', 'day', 'day', 'day']


**Image Annotations**

This relates to ground through, cant be used.

**GPS Coordinates (long, latt)**

In [6]:
coords = [(zod_frames[frame_id].metadata.latitude, zod_frames[frame_id].metadata.longitude) for frame_id in training_frames]
print(coords)

[(50.942598380571205, 6.935002170364342), (52.23059352385306, 20.999896372827468), (52.2305841982637, 21.069689396076246), (51.46754615052335, 7.005984963587919), (50.12259933340333, 8.740264282539721), (50.03671547267786, 19.940149527499187), (59.85306955881587, 17.63324022335248), (45.46774161948694, 9.155494476625615), (52.200754535135985, 21.011181088687994), (50.124889508176864, 8.745689140859351)]


**Environmental proxies**

Most relevant features. 

Road

In [7]:
road_type = [zod_frames[frame_id].metadata.road_type for frame_id in training_frames]
print(road_type)

['city', 'city', 'arterial-urban', 'arterial-urban', 'arterial-urban', 'arterial-urban', 'city', 'city', 'city', 'arterial-urban']


In [8]:
road_condition = [zod_frames[frame_id].metadata.road_condition for frame_id in training_frames]
print(road_condition)

['normal', 'wet', 'normal', 'wet', 'normal', 'normal', 'normal', 'normal', 'wet', 'normal']


Weather

In [9]:
weather = [zod_frames[frame_id].metadata.scraped_weather for frame_id in training_frames]
print(weather)

['cloudy', 'rain', 'partly-cloudy-night', 'rain', 'partly-cloudy-day', 'partly-cloudy-night', 'partly-cloudy-day', 'rain', 'rain', 'partly-cloudy-day']


Angle of Sun

In [10]:
solar_angle_elevation = [zod_frames[frame_id].metadata.solar_angle_elevation for frame_id in training_frames]
print(solar_angle_elevation)

[16.011264378275825, 32.23664620545043, -24.919594165615237, 10.267734194721413, 32.46558792539251, -26.281938274816014, 45.873164797571725, 30.96653709030604, 23.857360977858633, 32.42369451161603]


### Car Features

Features:
- Ego Motion: acceleartion, angular_rates, original_lat_long, poses, velocities, timestamp
- Calidation: lidars, cameras + metadata of camera as filepath, **time**, h, w

Ego Motion

List of 22 values. Too much info, need to compress. 

In [11]:
acc = [zod_frames[frame_id].ego_motion.accelerations for frame_id in training_frames]
print(acc)

[array([[ 1.14180016e-01, -3.80168843e-01, -1.02942694e+01],
       [-3.03773693e-03, -5.39875131e-01, -9.75608944e+00],
       [ 2.25912357e-02, -4.83826941e-01, -9.46551439e+00],
       [-5.69724340e-02, -5.21830525e-01, -9.72210376e+00],
       [ 2.08592375e-03, -5.20019729e-01, -9.72991922e+00],
       [ 8.52672484e-02, -3.01413204e-01, -9.70154955e+00],
       [-5.01919605e-02, -4.62126901e-01, -1.04672733e+01],
       [-2.82482350e-01, -8.73683084e-01, -9.75753423e+00],
       [-6.72319696e-02, -5.62980638e-01, -9.63852127e+00],
       [ 2.40529385e-01, -1.54019372e-01, -8.08611509e+00],
       [ 9.35542594e-02, -1.01796391e+00, -1.05445854e+01],
       [ 5.59825764e-02,  1.98364602e-01, -1.08625618e+01],
       [ 4.98817610e-02,  9.23018811e-02, -1.04648870e+01],
       [-2.22897909e-02,  2.27022025e-01, -9.60833578e+00],
       [-5.83888563e-02,  7.24125957e-02, -8.90351353e+00],
       [-1.88731111e-02,  6.06981117e-02, -9.48267239e+00],
       [ 9.25253344e-02,  3.49297928e-0

In [12]:
ang_rates = [zod_frames[frame_id].ego_motion.angular_rates for frame_id in training_frames]
print(acc)

[array([[ 1.14180016e-01, -3.80168843e-01, -1.02942694e+01],
       [-3.03773693e-03, -5.39875131e-01, -9.75608944e+00],
       [ 2.25912357e-02, -4.83826941e-01, -9.46551439e+00],
       [-5.69724340e-02, -5.21830525e-01, -9.72210376e+00],
       [ 2.08592375e-03, -5.20019729e-01, -9.72991922e+00],
       [ 8.52672484e-02, -3.01413204e-01, -9.70154955e+00],
       [-5.01919605e-02, -4.62126901e-01, -1.04672733e+01],
       [-2.82482350e-01, -8.73683084e-01, -9.75753423e+00],
       [-6.72319696e-02, -5.62980638e-01, -9.63852127e+00],
       [ 2.40529385e-01, -1.54019372e-01, -8.08611509e+00],
       [ 9.35542594e-02, -1.01796391e+00, -1.05445854e+01],
       [ 5.59825764e-02,  1.98364602e-01, -1.08625618e+01],
       [ 4.98817610e-02,  9.23018811e-02, -1.04648870e+01],
       [-2.22897909e-02,  2.27022025e-01, -9.60833578e+00],
       [-5.83888563e-02,  7.24125957e-02, -8.90351353e+00],
       [-1.88731111e-02,  6.06981117e-02, -9.48267239e+00],
       [ 9.25253344e-02,  3.49297928e-0

In [13]:
orig_lat_lon = [zod_frames[frame_id].ego_motion.origin_lat_lon for frame_id in training_frames]
print(acc)

[array([[ 1.14180016e-01, -3.80168843e-01, -1.02942694e+01],
       [-3.03773693e-03, -5.39875131e-01, -9.75608944e+00],
       [ 2.25912357e-02, -4.83826941e-01, -9.46551439e+00],
       [-5.69724340e-02, -5.21830525e-01, -9.72210376e+00],
       [ 2.08592375e-03, -5.20019729e-01, -9.72991922e+00],
       [ 8.52672484e-02, -3.01413204e-01, -9.70154955e+00],
       [-5.01919605e-02, -4.62126901e-01, -1.04672733e+01],
       [-2.82482350e-01, -8.73683084e-01, -9.75753423e+00],
       [-6.72319696e-02, -5.62980638e-01, -9.63852127e+00],
       [ 2.40529385e-01, -1.54019372e-01, -8.08611509e+00],
       [ 9.35542594e-02, -1.01796391e+00, -1.05445854e+01],
       [ 5.59825764e-02,  1.98364602e-01, -1.08625618e+01],
       [ 4.98817610e-02,  9.23018811e-02, -1.04648870e+01],
       [-2.22897909e-02,  2.27022025e-01, -9.60833578e+00],
       [-5.83888563e-02,  7.24125957e-02, -8.90351353e+00],
       [-1.88731111e-02,  6.06981117e-02, -9.48267239e+00],
       [ 9.25253344e-02,  3.49297928e-0

In poses, upper-left 3x3 matrix is the rotation matrix and last column contains the translation.


In [14]:
poses = [zod_frames[frame_id].ego_motion.poses for frame_id in training_frames]
print(acc)

[array([[ 1.14180016e-01, -3.80168843e-01, -1.02942694e+01],
       [-3.03773693e-03, -5.39875131e-01, -9.75608944e+00],
       [ 2.25912357e-02, -4.83826941e-01, -9.46551439e+00],
       [-5.69724340e-02, -5.21830525e-01, -9.72210376e+00],
       [ 2.08592375e-03, -5.20019729e-01, -9.72991922e+00],
       [ 8.52672484e-02, -3.01413204e-01, -9.70154955e+00],
       [-5.01919605e-02, -4.62126901e-01, -1.04672733e+01],
       [-2.82482350e-01, -8.73683084e-01, -9.75753423e+00],
       [-6.72319696e-02, -5.62980638e-01, -9.63852127e+00],
       [ 2.40529385e-01, -1.54019372e-01, -8.08611509e+00],
       [ 9.35542594e-02, -1.01796391e+00, -1.05445854e+01],
       [ 5.59825764e-02,  1.98364602e-01, -1.08625618e+01],
       [ 4.98817610e-02,  9.23018811e-02, -1.04648870e+01],
       [-2.22897909e-02,  2.27022025e-01, -9.60833578e+00],
       [-5.83888563e-02,  7.24125957e-02, -8.90351353e+00],
       [-1.88731111e-02,  6.06981117e-02, -9.48267239e+00],
       [ 9.25253344e-02,  3.49297928e-0

In [15]:
vel = [zod_frames[frame_id].ego_motion.velocities for frame_id in training_frames]
print(acc)

[array([[ 1.14180016e-01, -3.80168843e-01, -1.02942694e+01],
       [-3.03773693e-03, -5.39875131e-01, -9.75608944e+00],
       [ 2.25912357e-02, -4.83826941e-01, -9.46551439e+00],
       [-5.69724340e-02, -5.21830525e-01, -9.72210376e+00],
       [ 2.08592375e-03, -5.20019729e-01, -9.72991922e+00],
       [ 8.52672484e-02, -3.01413204e-01, -9.70154955e+00],
       [-5.01919605e-02, -4.62126901e-01, -1.04672733e+01],
       [-2.82482350e-01, -8.73683084e-01, -9.75753423e+00],
       [-6.72319696e-02, -5.62980638e-01, -9.63852127e+00],
       [ 2.40529385e-01, -1.54019372e-01, -8.08611509e+00],
       [ 9.35542594e-02, -1.01796391e+00, -1.05445854e+01],
       [ 5.59825764e-02,  1.98364602e-01, -1.08625618e+01],
       [ 4.98817610e-02,  9.23018811e-02, -1.04648870e+01],
       [-2.22897909e-02,  2.27022025e-01, -9.60833578e+00],
       [-5.83888563e-02,  7.24125957e-02, -8.90351353e+00],
       [-1.88731111e-02,  6.06981117e-02, -9.48267239e+00],
       [ 9.25253344e-02,  3.49297928e-0

In [16]:
timestamp = [zod_frames[frame_id].ego_motion.timestamps for frame_id in training_frames]
print(timestamp)

[array([1.60595189e+09, 1.60595189e+09, 1.60595189e+09, 1.60595189e+09,
       1.60595189e+09, 1.60595189e+09, 1.60595189e+09, 1.60595190e+09,
       1.60595190e+09, 1.60595190e+09, 1.60595190e+09, 1.60595190e+09,
       1.60595190e+09, 1.60595190e+09, 1.60595190e+09, 1.60595190e+09,
       1.60595190e+09, 1.60595190e+09, 1.60595190e+09, 1.60595190e+09,
       1.60595190e+09, 1.60595190e+09]), array([1.61996533e+09, 1.61996533e+09, 1.61996533e+09, 1.61996533e+09,
       1.61996533e+09, 1.61996533e+09, 1.61996533e+09, 1.61996533e+09,
       1.61996533e+09, 1.61996533e+09, 1.61996533e+09, 1.61996533e+09,
       1.61996533e+09, 1.61996533e+09, 1.61996533e+09, 1.61996533e+09,
       1.61996533e+09, 1.61996533e+09, 1.61996533e+09, 1.61996533e+09,
       1.61996533e+09, 1.61996533e+09]), array([1.61896064e+09, 1.61896064e+09, 1.61896064e+09, 1.61896064e+09,
       1.61896064e+09, 1.61896064e+09, 1.61896064e+09, 1.61896064e+09,
       1.61896064e+09, 1.61896064e+09, 1.61896064e+09, 1.61896064

**Calidations**

Camera

Also too much info

In [17]:
calid_cam = [zod_frames[frame_id].calibration.cameras for frame_id in training_frames]
print(calid_cam)

[{<Camera.FRONT: 'front'>: CameraCalibration(extrinsics=Pose(transform=array([[-2.14329725e-02, -5.45243113e-03,  9.99755419e-01,
         2.01530662e+00],
       [-9.99662687e-01, -1.45539035e-02, -2.15103581e-02,
         4.90213071e-04],
       [ 1.46676276e-02, -9.99879220e-01, -5.13865854e-03,
         1.12506001e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])), intrinsics=array([[1.85734965e+03, 0.00000000e+00, 1.91792462e+03, 0.00000000e+00],
       [0.00000000e+00, 1.85734965e+03, 1.11514928e+03, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.00000000e+00, 0.00000000e+00]]), distortion=array([-0.00592982, -0.05365763,  0.06591502, -0.02476903]), undistortion=array([ 0.00514188,  0.0593341 , -0.07306569,  0.02798947]), image_dimensions=array([3848, 2168]), field_of_view=array([121.16048371,  67.28731745]))}, {<Camera.FRONT: 'front'>: CameraCalibration(extrinsics=Pose(transform=array([[ 0.00786083,  0.02827075,  0.99956939, 

We also have a datatime of when was the image recorded, meaning we have info of time of day

In [18]:
datetime = [str(zod_frames[frame_id].info.keyframe_time) for frame_id in training_frames]
print(datetime)

['2020-11-21 09:44:55.445654+00:00', '2021-05-02 14:22:06.780681+00:00', '2021-04-20 23:17:20.001688+00:00', '2020-10-29 14:47:10.887313+00:00', '2020-10-10 10:42:09.883075+00:00', '2021-04-20 23:41:36.002429+00:00', '2020-05-06 11:41:57.779576+00:00', '2020-07-24 07:08:56.778861+00:00', '2021-04-15 14:52:36.885711+00:00', '2020-10-10 10:40:46.448060+00:00']


In [38]:
datetime_hour = [str((zod_frames[frame_id].info.keyframe_time).replace(minute=0, second=0, microsecond=0)) for frame_id in training_frames]
print(datetime_hour)

['2020-11-21 09:00:00+00:00', '2021-05-02 14:00:00+00:00', '2021-04-20 23:00:00+00:00', '2020-10-29 14:00:00+00:00', '2020-10-10 10:00:00+00:00', '2021-04-20 23:00:00+00:00', '2020-05-06 11:00:00+00:00', '2020-07-24 07:00:00+00:00', '2021-04-15 14:00:00+00:00', '2020-10-10 10:00:00+00:00']


In [19]:
month = [zod_frames[frame_id].info.keyframe_time.month for frame_id in training_frames]
print(month)

[11, 5, 4, 10, 10, 4, 5, 7, 4, 10]


Hour in UTC00, will need to adjust for each country.

In [20]:
utc_offsets = {
    'PL': +1,  # Poland 
    'SE': +1,  # Sweden 
    'IT': +1,  # Italy 
    'DE': +1,  # Germany 
}

In [21]:
hour = [zod_frames[frame_id].info.keyframe_time.hour + utc_offsets[country] for frame_id, country in zip(training_frames, country)]
print(hour)

[10, 15, 24, 15, 11, 24, 12, 8, 15, 11]


### Image

**Metrics**

Use No-Reference (NR) methods. https://piq.readthedocs.io/en/latest/overview.html
- BRISQUE: quality score based on natural scene statistics, heavy to compute. 
- CLIP-IQA: weakly supervised IQA that leverages CLIP’s multimodal embeddings. Even more heavy in compute and does not know if it generalizes. 

**Embeddings**

## Weather API

In [30]:
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": coords[0][0],
	"longitude": coords[0][1],
	"start_date": datetime[0].split(" ")[0],
	"end_date": datetime[0].split(" ")[0],
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "weather_code", "pressure_msl", "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "et0_fao_evapotranspiration"],
	"timezone": "UTC"
}
print(params)

{'latitude': 50.942598380571205, 'longitude': 6.935002170364342, 'start_date': '2020-11-21', 'end_date': '2020-11-21', 'hourly': ['temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'pressure_msl', 'surface_pressure', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'et0_fao_evapotranspiration'], 'timezone': 'UTC'}


In [31]:
responses = openmeteo.weather_api(url, params=params)

response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to UTC+0 {response.UtcOffsetSeconds()} s")

Coordinates 50.93145751953125°N 6.910714149475098°E
Elevation 57.0 m asl
Timezone NoneNone
Timezone difference to UTC+0 0 s


In [43]:
hourly = response.Hourly()

hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_rain = hourly.Variables(5).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(6).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(7).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(8).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(9).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(10).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(11).ValuesAsNumpy()
hourly_cloud_cover_low = hourly.Variables(12).ValuesAsNumpy()
hourly_cloud_cover_mid = hourly.Variables(13).ValuesAsNumpy()
hourly_et0_fao_evapotranspiration = hourly.Variables(14).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["weather_code"] = hourly_weather_code
hourly_data["pressure_msl"] = hourly_pressure_msl
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
hourly_data["et0_fao_evapotranspiration"] = hourly_et0_fao_evapotranspiration

hourly_dataframe = pd.DataFrame(data = hourly_data)
print(hourly_dataframe.query(f"date == '{datetime_hour[0]}'"))

                       date  temperature_2m  relative_humidity_2m  \
9 2020-11-21 09:00:00+00:00          6.0695             73.667458   

   dew_point_2m  apparent_temperature  precipitation  rain  snowfall  \
9        1.7195              2.448959            0.0   0.0       0.0   

   snow_depth  weather_code  pressure_msl  surface_pressure  cloud_cover  \
9         0.0           3.0   1032.099976       1024.932129        100.0   

   cloud_cover_low  cloud_cover_mid  et0_fao_evapotranspiration  
9            100.0             74.0                    0.044103  


/tmp/ipykernel_3075253/2342237315.py:43: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  print(hourly_dataframe.query(f"date == '{datetime_hour[0]}'"))


# Final Tabular Data

In [23]:
data_dict = {  
    "country": country, 
    "time_of_day": time_of_day, 
    "coords": coords, 
    "road_type": road_type,
    "road_condition": road_condition, 
    "weather": weather,
    "solar_angle_elevation": solar_angle_elevation,
    "month": month,  
    "hour": hour,
}

df = pd.DataFrame(data=data_dict, index= frames_id)
print(df.head())
df.to_csv("../data/metafeatures.csv")

       country time_of_day                                   coords  \
009158      DE         day  (50.942598380571205, 6.935002170364342)   
029229      PL         day  (52.23059352385306, 20.999896372827468)   
082291      PL       night   (52.2305841982637, 21.069689396076246)   
087912      DE         day   (51.46754615052335, 7.005984963587919)   
065838      DE         day   (50.12259933340333, 8.740264282539721)   

             road_type road_condition              weather  \
009158            city         normal               cloudy   
029229            city            wet                 rain   
082291  arterial-urban         normal  partly-cloudy-night   
087912  arterial-urban            wet                 rain   
065838  arterial-urban         normal    partly-cloudy-day   

        solar_angle_elevation  month  hour  
009158              16.011264     11    10  
029229              32.236646      5    15  
082291             -24.919594      4    24  
087912              

In [24]:
columns = df.columns
unique_values = [f"{column}:" + str(df[column].unique()) for column in columns]
print("\n".join(unique_values))

country:['DE' 'PL' 'SE' 'IT']
time_of_day:['day' 'night']
coords:[(50.942598380571205, 6.935002170364342)
 (52.23059352385306, 20.999896372827468)
 (52.2305841982637, 21.069689396076246)
 (51.46754615052335, 7.005984963587919)
 (50.12259933340333, 8.740264282539721)
 (50.03671547267786, 19.940149527499187)
 (59.85306955881587, 17.63324022335248)
 (45.46774161948694, 9.155494476625615)
 (52.200754535135985, 21.011181088687994)
 (50.124889508176864, 8.745689140859351)]
road_type:['city' 'arterial-urban']
road_condition:['normal' 'wet']
weather:['cloudy' 'rain' 'partly-cloudy-night' 'partly-cloudy-day']
solar_angle_elevation:[ 16.01126438  32.23664621 -24.91959417  10.26773419  32.46558793
 -26.28193827  45.8731648   30.96653709  23.85736098  32.42369451]
month:[11  5  4 10  7]
hour:[10 15 24 11 12  8]
